In [18]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool

In [20]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

In [21]:
multiply.invoke(
    {
        "a" : 10,
        "b" : 3
    }
)

30

In [22]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [23]:
# Creating agent
from langchain.agents import create_agent
agent = create_agent(
    model = llm,
    tools = [multiply],
    system_prompt = "You are a helpful math assistant. Use the multiply tool when multiplication is needed"
)

In [27]:
response = agent.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content" : "What is 125 multiplied by 8?"
            }
        ]
    }
)

In [38]:
response

{'messages': [HumanMessage(content='What is 125 multiplied by 8?', additional_kwargs={}, response_metadata={}, id='b5f6be85-2052-49b7-a7de-495cb9afca6c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 148, 'total_tokens': 190, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 16, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.394525388, 'prompt_time': 0.007400486, 'completion_time': 0.044106009, 'total_time': 0.051506495}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e11c8bbf69', 'id': 'chatcmpl-5f81fe66-a07d-48a8-9cd8-eebe78d4da8b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13147-dd9c-7c61-8663-953926b6e03a-0', tool_calls=[{'name': 'multiply', 'args': {'a': 125, 'b': 8}, 'id': 'fc_21404272-d877-4a37-88fc-5

In [37]:
response["messages"][-2].content

'1000'

In [40]:
# Earlier we were doing the task manually, we were doing these three things after creating the Tool - Tool binding, Tool calling and Tool execution.
# But now these three tasks (Tool binding, Tool calling and Tool execution) were automatically done by an agent which reduces manual work

In [41]:
for message in response["messages"]:
    print("Type:", type(message).__name__)
    print("Content:", message.content)
    print("-" * 50)

Type: HumanMessage
Content: What is 125 multiplied by 8?
--------------------------------------------------
Type: AIMessage
Content: 
--------------------------------------------------
Type: ToolMessage
Content: 1000
--------------------------------------------------
Type: AIMessage
Content: \(125 \times 8 = 1{,}000\).
--------------------------------------------------


In [42]:
for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        print(message.tool_calls)

[{'name': 'multiply', 'args': {'a': 125, 'b': 8}, 'id': 'fc_21404272-d877-4a37-88fc-56452ffbad93', 'type': 'tool_call'}]


### Adding another tool

In [44]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

In [45]:
agent = create_agent(
    model=llm,
    tools=[multiply, add],
    system_prompt=(
        "You are a helpful math assistant. "
        "Use the appropriate tool for addition or multiplication."
    )
)

In [47]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 45 plus 55?"
        }
    ]
})

In [48]:
response

{'messages': [HumanMessage(content='What is 45 plus 55?', additional_kwargs={}, response_metadata={}, id='02e23298-19cc-4863-b498-e5ead8cb2c55'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 169, 'total_tokens': 202, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 7, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.302515749, 'prompt_time': 0.009607211, 'completion_time': 0.034450278, 'total_time': 0.044057489}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'id': 'chatcmpl-8df7bf05-d417-4384-a286-3fb0f551adc4', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1314f-3054-74f2-bf80-45467dec576e-0', tool_calls=[{'name': 'add', 'args': {'a': 45, 'b': 55}, 'id': 'fc_52dd7991-f717-41c7-964d-509e66061c4c', '

In [50]:
response["messages"]

[HumanMessage(content='What is 45 plus 55?', additional_kwargs={}, response_metadata={}, id='02e23298-19cc-4863-b498-e5ead8cb2c55'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 169, 'total_tokens': 202, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 7, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.302515749, 'prompt_time': 0.009607211, 'completion_time': 0.034450278, 'total_time': 0.044057489}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'id': 'chatcmpl-8df7bf05-d417-4384-a286-3fb0f551adc4', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1314f-3054-74f2-bf80-45467dec576e-0', tool_calls=[{'name': 'add', 'args': {'a': 45, 'b': 55}, 'id': 'fc_52dd7991-f717-41c7-964d-509e66061c4c', 'type': 'tool_c

In [53]:
response["messages"][-2].content

'100'

In [55]:
for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print("Selected tool:", call["name"])
            print("Arguments:", call["args"])

Selected tool: add
Arguments: {'a': 45, 'b': 55}


### Doing more than one operation in a single query

In [56]:
response = agent.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content" : "Multiply 12 by 5, and then add 50 to it"
            }
        ]
    }
)

In [59]:
response["messages"]

[HumanMessage(content='Multiply 12 by 5, and then add 50 to it', additional_kwargs={}, response_metadata={}, id='1d472896-c562-40b3-9db9-e0062191296d'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 175, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.303538752, 'prompt_time': 0.015140398, 'completion_time': 0.05940759, 'total_time': 0.074547988}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'id': 'chatcmpl-c37523b2-c89e-4137-af6b-e206b0c23a9b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13153-557e-7f02-86cb-2f68e49addbc-0', tool_calls=[{'name': 'multiply', 'args': {'a': 12, 'b': 5}, 'id': 'fc_6a623240-f0d0-46d8-9a27-516227

In [60]:
response["messages"][-1].content

'110'

In [61]:
for message in response["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("Tool:", call["name"])
        print("Arguments:", call["args"])
        print()

Tool: multiply
Arguments: {'a': 12, 'b': 5}

Tool: add
Arguments: {'a': 60, 'b': 50}

